In [1]:
import re

def sms_scam_check(text):
    t = text.lower()
    rules = [
        (r"https?://|bit\.ly|tinyurl|www\.", 30, "Contains a suspicious link"),
        (r"\botp\b|\bpin\b|\bcvv\b", 25, "Asks for OTP/PIN/CVV"),
        (r"blocked|suspended|kyc|expire|deactivat", 25, "Fake bank alert wording"),
        (r"lottery|winner|won |prize|lucky draw|reward", 25, "Lottery/prize bait"),
        (r"urgent|immediately|within 24|act now|last chance", 15, "Urgency pressure"),
        (r"processing fee|send money|transfer", 20, "Asks for money"),
    ]
    score, reasons = 0, []
    for pattern, pts, why in rules:
        if re.search(pattern, t):
            score += pts
            reasons.append(why)
    score = min(score, 100)
    level = "HIGH" if score >= 60 else "MEDIUM" if score >= 30 else "LOW"
    return {"risk_score": score, "level": level, "reasons": reasons}

print(sms_scam_check("Your account is blocked! Share OTP now or click http://bit.ly/kyc"))

{'risk_score': 80, 'level': 'HIGH', 'reasons': ['Contains a suspicious link', 'Asks for OTP/PIN/CVV', 'Fake bank alert wording']}


In [2]:
import pandas as pd, numpy as np
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import classification_report, confusion_matrix

np.random.seed(42)
n = 100000
df = pd.DataFrame({
    "amount": np.random.exponential(80, n),
    "hour": np.random.randint(0, 24, n),
    "avg_spend_30d": np.random.normal(80, 20, n).clip(5),
    "txns_last_24h": np.random.poisson(3, n),
    "new_device": np.random.binomial(1, 0.05, n),
})
df["is_fraud"] = (
    ((df.amount > 4*df.avg_spend_30d) & (df.new_device == 1)) |
    ((df.amount > 500) & df.hour.isin([0,1,2,3])) |
    (df.txns_last_24h > 10)
).astype(int)

X = df.drop("is_fraud", axis=1)
y = df["is_fraud"]
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, stratify=y, random_state=42)

model = RandomForestClassifier(
    n_estimators=200, class_weight="balanced", random_state=42)
model.fit(X_train, y_train)

pred = model.predict(X_test)
print(classification_report(y_test, pred))
print(confusion_matrix(y_test, pred))

              precision    recall  f1-score   support

           0       1.00      1.00      1.00     19956
           1       1.00      1.00      1.00        44

    accuracy                           1.00     20000
   macro avg       1.00      1.00      1.00     20000
weighted avg       1.00      1.00      1.00     20000

[[19956     0]
 [    0    44]]


In [3]:
def transaction_check(txn):
    row = pd.DataFrame([txn])[X.columns]
    prob = model.predict_proba(row)[0][1]
    reasons = []
    if txn["amount"] > 4 * txn["avg_spend_30d"]:
        reasons.append(f"Amount is {txn['amount']/txn['avg_spend_30d']:.1f}x your 30-day average")
    if txn["txns_last_24h"] > 8:
        reasons.append("Unusually many transactions in 24 hours")
    if txn["new_device"] == 1:
        reasons.append("Payment from a new device")
    if txn["hour"] in [0, 1, 2, 3] and txn["amount"] > 300:
        reasons.append("Large payment at night")
    return prob, reasons

print(transaction_check({"amount": 2500, "hour": 2,
    "avg_spend_30d": 60, "txns_last_24h": 4, "new_device": 1}))

(np.float64(0.88), ['Amount is 41.7x your 30-day average', 'Payment from a new device', 'Large payment at night'])


In [4]:
import re

reported_upi = {
    "winner.prize@upi": "Reported 14 times: lottery scam",
    "kyc.update@paytm": "Reported 9 times: fake KYC",
}
verified_upi = {"amazon@apl", "swiggy@icici"}

def upi_check(upi_id):
    upi_id = upi_id.strip().lower()
    if not re.fullmatch(r"[a-z0-9.\-_]{2,}@[a-z]{2,}", upi_id):
        return {"status": "INVALID", "warning": "Not a valid UPI ID format"}
    if upi_id in reported_upi:
        return {"status": "REPORTED", "warning": reported_upi[upi_id]}
    if upi_id in verified_upi:
        return {"status": "VERIFIED", "warning": "Known merchant"}
    return {"status": "UNVERIFIED", "warning": "Not in the database. Verify before paying."}

print(upi_check("winner.prize@upi"))

{'status': 'REPORTED', 'warning': 'Reported 14 times: lottery scam'}


In [5]:
import random, time

class DemoOTP:
    def __init__(self, validity=60):
        self.validity, self.code, self.created, self.tries = validity, None, 0, 0

    def generate(self):
        self.code = str(random.randint(100000, 999999))
        self.created, self.tries = time.time(), 0
        return self.code  # shown on screen for the demo

    def verify(self, entered):
        if self.code is None:
            return "No OTP generated"
        if time.time() - self.created > self.validity:
            return "❌ OTP expired"
        self.tries += 1
        if self.tries > 3:
            return "🔒 Too many attempts"
        return "✅ Verified" if entered == self.code else "❌ Wrong OTP"

otp = DemoOTP()
code = otp.generate()
print("Demo OTP:", code)
print(otp.verify("000000"))
print(otp.verify(code))

Demo OTP: 773262
❌ Wrong OTP
✅ Verified


In [6]:
def fraud_alert(txn, upi_id=None, sms=None):
    prob, reasons = transaction_check(txn)
    score = float(prob) * 100
    if upi_id:
        u = upi_check(upi_id)
        if u["status"] == "REPORTED":
            score = max(score, 90)
            reasons.append("UPI ID: " + u["warning"])
        elif u["status"] in ("UNVERIFIED", "INVALID"):
            score += 15
            reasons.append("UPI ID: " + u["warning"])
    if sms:
        s = sms_scam_check(sms)
        if s["level"] != "LOW":
            score = max(score, s["risk_score"])
            reasons += ["SMS: " + r for r in s["reasons"]]
    score = min(score, 100)
    level = "HIGH" if score >= 60 else "MEDIUM" if score >= 30 else "LOW"
    steps = {
        "HIGH": ["Do NOT proceed", "Never share your OTP", "Call your bank's official number"],
        "MEDIUM": ["Verify the receiver first", "Check the amount carefully"],
        "LOW": ["Looks normal. Stay alert."],
    }[level]
    print(f"Risk: {level} ({score:.0f}/100)")
    print("Why:", reasons or ["No warning signs"])
    print("Next steps:", steps)

print("--- Risky example ---")
fraud_alert(
    {"amount": 2500, "hour": 2, "avg_spend_30d": 60, "txns_last_24h": 4, "new_device": 1},
    upi_id="winner.prize@upi",
    sms="Urgent! You won a prize. Pay processing fee at http://bit.ly/x",
)

print("\n--- Safe example ---")
fraud_alert(
    {"amount": 50, "hour": 14, "avg_spend_30d": 80, "txns_last_24h": 2, "new_device": 0},
    upi_id="amazon@apl",
)

--- Risky example ---
Risk: HIGH (90/100)
Why: ['Amount is 41.7x your 30-day average', 'Payment from a new device', 'Large payment at night', 'UPI ID: Reported 14 times: lottery scam', 'SMS: Contains a suspicious link', 'SMS: Lottery/prize bait', 'SMS: Urgency pressure', 'SMS: Asks for money']
Next steps: ['Do NOT proceed', 'Never share your OTP', "Call your bank's official number"]

--- Safe example ---
Risk: LOW (0/100)
Why: ['No warning signs']
Next steps: ['Looks normal. Stay alert.']


In [8]:
!pip install -q gradio
import gradio as gr

def app(amount, hour, avg_spend, txns, new_device, upi_id, sms):
    txn = {"amount": amount, "hour": int(hour), "avg_spend_30d": avg_spend,
           "txns_last_24h": int(txns), "new_device": int(new_device)}
    prob, reasons = transaction_check(txn)
    score = float(prob) * 100
    if upi_id:
        u = upi_check(upi_id)
        if u["status"] == "REPORTED":
            score = max(score, 90); reasons.append("UPI ID: " + u["warning"])
        elif u["status"] in ("UNVERIFIED", "INVALID"):
            score += 15; reasons.append("UPI ID: " + u["warning"])
    if sms:
        s = sms_scam_check(sms)
        if s["level"] != "LOW":
            score = max(score, s["risk_score"])
            reasons += ["SMS: " + r for r in s["reasons"]]
    score = min(score, 100)
    level = "HIGH" if score >= 60 else "MEDIUM" if score >= 30 else "LOW"
    steps = {"HIGH": "Do NOT proceed. Never share your OTP. Call your bank's official number.",
             "MEDIUM": "Verify the receiver first. Check the amount carefully.",
             "LOW": "Looks normal. Stay alert."}[level]
    why = "\n".join("- " + r for r in reasons) or "No warning signs"
    return f"Risk: {level} ({score:.0f}/100)", why, steps

gr.Interface(
    fn=app,
    inputs=[
        gr.Number(label="Amount", value=2500),
        gr.Slider(0, 23, step=1, value=2, label="Hour of day"),
        gr.Number(label="Average spend (30 days)", value=60),
        gr.Number(label="Transactions in last 24h", value=4),
        gr.Radio([0, 1], label="New device? (1 = yes)", value=1),
        gr.Textbox(label="UPI ID (optional)", value="winner.prize@upi"),
        gr.Textbox(label="SMS text (optional)", lines=3),
    ],
    outputs=[gr.Textbox(label="Risk level"),
             gr.Textbox(label="Why", lines=5),
             gr.Textbox(label="Next steps")],
    title="FraudShield AI",
    description="Check a transaction, UPI ID or SMS for fraud risk.",
).launch(share=True)

Colab notebook detected. To show errors in colab notebook, set debug=True in launch()
* Running on public URL: https://a474b1b7644f720b45.gradio.live

This share link is temporary and will last for up to 1 week (best effort). For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)
